# 📚 Introduction to Retrieval Augmented Generation with LangChain 🦜🔗

In this notebook you'll learn how to use LangChain for Retrieval Augmented Generation.

We will use an LLM to answer questions about our own documents!

## ⚙️ Setup

👉 Run the cell below to import a couple of basic libraries.

In [2]:
%load_ext autoreload
%autoreload 2
import os
from pprint import pprint
from IPython.display import Markdown

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


👉 Run the cell below to load our API key again:

In [3]:
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

True

## 📚 Why RAG?

An LLM on its own can respond questions about everything it has learned.

That has a couple of drawbacks:
- The training data comes from the past and is not updated with the most recent data.
- It only knows the data it was trained on.

We want to use an LLM to work with our own data. That is where RAG, or Retrieval-Augmented Generation steps in.

1. **Retrieval-Augmented Generation (RAG)** combines a language model with a document retriever to enhance factual accuracy.
2. **It retrieves relevant external documents** (e.g., from a knowledge base) before generating responses.
3. **The language model uses both the prompt and retrieved context** to produce more informed and grounded outputs.

## 🇪🇺 Context

In this challenge, we'll work with documents from the European Parliament.

Imagine you're a reporter, and you want to know what has been said about a certain topic during the European Parliament's plenary sessions. Those sessions take place 12 times a year in Strassbourg, and last 4 days. Transcriptions of the sessions are available on the EP's website.

You definitely don't want to go ploughing through all those transcripts. So, let's leverage RAG to make our life easier!

This is good data to work with, because at all times we can take brand new data to test it out.

## 📘 Let's get the data

1. Head to the [EP's website](https://www.europarl.europa.eu/plenary/en/debates-video.html). 
1. That will lead you to the most recent plenary session.
1. Under the first date, click on `HTML` in "▶️ Verbatim reports HTML".
1. Scroll to the bottom of the page, and download the PDF file at the bottom.
1. Save the file in the `data` folder.

We'll start with one document, but you can already download the same for a couple of other days for later.

Have a look at the document. How many pages does it have? Quickly scroll through the document to get a feel for it.


## 🔢 Embedding documents

Embedding documents means that we will translate whole documents, or chunks of documents, into vectors.

LangChain🦜🔗 will be very helpful again.

Let's instantiate an embedder and try it out. Because we're using Gemini as our LLM, let's stick to Google's text embedders.

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

👉 Try the embedder's `.embed_query()` to embed a simple piece of text.

In [6]:
# Embed a text like "What is the capital of France?" and save it to a variable `sample_embedding`

# YOUR CODE HERE
sample_embedding = embeddings.embed_query('What is the capital of France?" and save it to a variable')

sample_embedding

[-0.027229346334934235,
 -0.015511598438024521,
 0.021039964631199837,
 -0.05218672379851341,
 -0.025438839569687843,
 -0.002378966426476836,
 -0.005148028023540974,
 -0.015525538474321365,
 0.039580971002578735,
 -0.01334705762565136,
 -0.045504361391067505,
 -0.017328543588519096,
 -0.01995193399488926,
 0.005634812638163567,
 0.13476645946502686,
 -0.018625864759087563,
 -0.012344160117208958,
 0.006020617671310902,
 0.027187524363398552,
 -0.033001914620399475,
 -0.02962024323642254,
 0.025738386437296867,
 0.02312626503407955,
 -0.023030057549476624,
 0.018491530790925026,
 -0.002115681767463684,
 0.0175496656447649,
 0.00043203195673413575,
 0.011923914775252342,
 0.028090378269553185,
 0.012300968170166016,
 -0.014246553182601929,
 0.0026565594598650932,
 0.002568403258919716,
 -0.00512389000505209,
 -0.0012023575836792588,
 0.01049239281564951,
 -0.015889542177319527,
 -0.006379121448844671,
 0.0030723302625119686,
 -0.01939206011593342,
 -0.013920694589614868,
 0.0198533833026

👉 Take the time to explore this `sample_embedding`. What does it look like? What's its type? What is the embedding size?

In [11]:
# YOUR CODE HERE
type(sample_embedding)
len(sample_embedding)
sample_embedding[:10]

[-0.027229346334934235,
 -0.015511598438024521,
 0.021039964631199837,
 -0.05218672379851341,
 -0.025438839569687843,
 -0.002378966426476836,
 -0.005148028023540974,
 -0.015525538474321365,
 0.039580971002578735,
 -0.01334705762565136]

## 💾 Load our real data from PDF

Now we know what an embedding looks like, it's time to get working with our real data.

👉 Head to the [LangChain documentation](https://docs.langchain.com/oss/python/integrations/document_loaders/index#pdfs), and find out how you can load a PDF using PyPDF.

👉 Then go ahead and load one of the PDFs you downloaded before.

In [15]:
# YOUR CODE HERE
from langchain_community.document_loaders import PyPDFLoader

file_path = 'data/CRE-10-2026-05-18_EN.pdf'

loader = PyPDFLoader(file_path)
doc = loader.load()

In [17]:
doc

[Document(metadata={'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '496443', 'dmxml.render.traceid': '6a0c9630e3d2f41712cb1448b203bc38', 'uid': 'eu.europa.europarl-DIN1-2026-0000144886_01.00-xm-01.00_text-xml', 'source': 'data/CRE-10-2026-05-18_EN.pdf', 'total_pages': 123, 'page': 0, 'page_label': '1'}, page_content='2024-2029 \n \n \nПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA \nACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA \nDOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE \nFULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI \nAUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN \nISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD \nΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES \nVERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR \nCOMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ ZÁPIS Z ROZPRÁV \nTUARASCÁIL FOC

👉 Explore the `pages`:
- What is its data type?
- How many pages do you have?
- What is the type of one page?
- How can you access the content of one page?
- How many characters does the full document have?
- What is in the `metadata` of a page?

In [ ]:
# YOUR CODE HERE
print(type(doc))
print(type(doc[0]))
print(len(doc))
print(doc[0].metadata)

<class 'list'>
<class 'langchain_core.documents.base.Document'>
123
{'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '496443', 'dmxml.render.traceid': '6a0c9630e3d2f41712cb1448b203bc38', 'uid': 'eu.europa.europarl-DIN1-2026-0000144886_01.00-xm-01.00_text-xml', 'source': 'data/CRE-10-2026-05-18_EN.pdf', 'total_pages': 123, 'page': 0, 'page_label': '1'}


## ✂️ Split our data

Our complete document is too long to be embedded. Our text embedder can take inputs up to 2.048 tokens. For Gemini models that is about 8.196 characters (4 characters per token).

So we want to split our document in smaller chunks.

We already have a bunch of pages we could work with. But page ends are a bit arbitrary: they usually appear in the middle of a sentence.

Also, there is no overlap between the pages. So the first line of a page misses all context before. It's better to split the full text with a bit of overlap.

First, we'll load the PDF again, this time without splitting it.

In [23]:
loader = PyPDFLoader(file_path, mode='single')
pdf = loader.load()
pdf_text = pdf[0].page_content
len(pdf_text)
pdf[0]

Document(metadata={'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '496443', 'dmxml.render.traceid': '6a0c9630e3d2f41712cb1448b203bc38', 'uid': 'eu.europa.europarl-DIN1-2026-0000144886_01.00-xm-01.00_text-xml', 'source': 'data/CRE-10-2026-05-18_EN.pdf', 'total_pages': 123}, page_content='2024-2029 \n \n \nПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA \nACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA \nDOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE \nFULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI \nAUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN \nISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD \nΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES \nVERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR \nCOMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ ZÁPIS Z ROZPRÁV \nTUARASCÁIL FOCAL AR FHOCAL NA N-IMEACHTAÍ  DO

Now that we have our whole PDF as one document, we can split it in chunks in a smarter way.

👉 Again, head over to the [LangChain documentation on "Splitting recursively"](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter) and find out how to split our `pdf` _documents_ into chunks (called `documents` in LangChain).

Split it in chunks of 2_000 characters (that's about half a page in our case) with an overlap of 400. You can experiment with other values if you want.

Use the `RecursiveCharacterTextSplitter`'s `.split_documents()` method: this method takes a document as input, and outputs splitted documents.

In [27]:
# YOUR CODE HERE
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=2000,
    chunk_overlap=400,
)

all_splits = text_splitter.split_documents(pdf)

👉 Inspect `all_splits`:
- What is its data type?
- How many splits do you have?
- What is the type of one split?
- How can you access the content of one split?
- How many characters do we have in total now?
- What is in the `metadata` of a split?

In [34]:
# YOUR CODE HERE
print(type(all_splits))
print(len(all_splits))
print(type(all_splits[0]))
print(all_splits[0])
print(all_splits[0].page_content)
print(all_splits[0].metadata)

<class 'list'>
234
<class 'langchain_core.documents.base.Document'>
page_content='2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE 
FULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI 
AUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN 
ISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD 
ΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES 
VERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR 
COMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ ZÁPIS Z ROZPRÁV 
TUARASCÁIL FOCAL AR FHOCAL NA N-IMEACHTAÍ  DOBESEDNI ZAPISI RAZPRAV 
DOSLOVNO IZVJEŠĆE  SANATARKAT ISTUNTOSELOSTUKSET 
RESOCONTO INTEGRALE DELLE DISCUSSIONI  FULLSTÄNDIGT FÖRHANDLINGSREFERAT 
 
 
Понеделник - Lunes - Pondělí - Mandag - Montag - Esmaspäev - Δευτέρα - Monday 
Lundi - Dé Luain - Ponedjeljak - Lunedì - Pirmdiena - Pirmadienis - Hétfő - It-Tnejn 
Maandag - 

## 🗄️ Bring it all together: embed and store our documents in a vector store

We have:
- An embedder
- A loader to load the data
- A text splitter to split our document into documents

What's missing?

We can embed our documents, but we want to store them somewhere. That's where a vector store comes in: it allows us to save:
- the document (the chunk),
- its embedding,
- its metadata.

In a next step we'll then be able to retrieve documents efficiently.

👉 Check the [LangChain documentation on "Vector stores"](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) to see how you can create an `InMemoryVectorStore`.

In [35]:
# Import the necessary libraries

# YOUR CODE HERE
from langchain_core.vectorstores import InMemoryVectorStore
# Create an in-memory vector store using the embedder `embeddings` we created earlier

# YOUR CODE HERE
vector_store = InMemoryVectorStore(embeddings)
# Add the `all_splits` to the vector store and store the result in a variable called `document_ids`

# YOUR CODE HERE
document_ids = vector_store.add_documents(documents=all_splits)

In [36]:
# Have a look at the first 3 document IDs

# YOUR CODE HERE
document_ids[:3]

['e78fcd9b-cbff-4404-9843-c41da701f895',
 '3126bb92-f39f-46fd-8a60-3bd8a73da133',
 '273b8644-beac-49e3-bbc0-12f6c0e21407']

In [37]:
# Use the vector store's `get_by_ids` method. You have to give it a list of document IDs.

# YOUR CODE HERE
vector_store.get_by_ids(document_ids[:3])

[Document(id='e78fcd9b-cbff-4404-9843-c41da701f895', metadata={'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '496443', 'dmxml.render.traceid': '6a0c9630e3d2f41712cb1448b203bc38', 'uid': 'eu.europa.europarl-DIN1-2026-0000144886_01.00-xm-01.00_text-xml', 'source': 'data/CRE-10-2026-05-18_EN.pdf', 'total_pages': 123}, page_content='2024-2029 \n \n \nПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA \nACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA \nDOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE \nFULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI \nAUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN \nISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD \nΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES \nVERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR \nCOMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ ZÁPIS Z ROZPRÁV \nT

👉 How can you access a vector store's document's content and metadata?

In [39]:
# YOUR CODE HERE
stored_docs = vector_store.get_by_ids(document_ids[:3])

print(stored_docs[0].page_content[:1000])

print(stored_docs[0].metadata)

2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE 
FULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI 
AUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN 
ISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD 
ΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES 
VERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR 
COMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ ZÁPIS Z ROZPRÁV 
TUARASCÁIL FOCAL AR FHOCAL NA N-IMEACHTAÍ  DOBESEDNI ZAPISI RAZPRAV 
DOSLOVNO IZVJEŠĆE  SANATARKAT ISTUNTOSELOSTUKSET 
RESOCONTO INTEGRALE DELLE DISCUSSIONI  FULLSTÄNDIGT FÖRHANDLINGSREFERAT 
 
 
Понеделник - Lunes - Pondělí - Mandag - Montag - Esmaspäev - Δευτέρα - Monday 
Lundi - Dé Luain - Ponedjeljak - Lunedì - Pirmdiena - Pirmadienis - Hétfő - It-Tnejn 
Maandag - Poniedziałek - Segunda-feira - Luni - Pondelok - Ponedeljek - Maanantai - Måndag 


## 🔎 Use the vector store to retrieve similar documents

Now that we embedded the documents, we can use the vector store to retrieve similar documents.

👉 Check in the [LangChain documentation on "Vector stores"](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) how that works.

Use a query, e.g. "Summarize the discussion on agricultural policy.", and find the most similar documents. You can also specify the number of documents to retrieve.

In [ ]:
# Save your question into a variable called `query`

# YOUR CODE HERE
query = 'Summerize the discussion about fishing'
# Use the vector store to find similar documents to the query. Store the result in a variable called `retrieved_docs`

# YOUR CODE HERE
retrieved_docs = vector_store.similarity_search(query)


This concludes the so-called "Retrieval" part of RAG: we can now find the documents that are the most similar to our query.

Most of the work is done now!

## 💬 Generate an answer to our question

So far we only used an **embedding model** to enable us to retrieve the most similar documents.

Now, we will use a generative LLM to get an answer to our question: we'll feed it with our retrieved documents, and our question.

The most rudimentary way to do this would be to concatenate all our inputs together, add our question, and see the result.

Let's give it a try.

👉 First instantiate an LLM like in the previous challenges.

In [45]:
# YOUR CODE HERE
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

Then create a rudimentary prompt:

In [46]:
prompt = '\n\n'.join([doc.page_content for doc in retrieved_docs])
prompt += "\n\n" + query

👉 Now use the prompt:

In [47]:
# YOUR CODE HERE
response = llm.invoke(prompt)

Markdown(response.content)

The discussion revolves heavily around the state of the Baltic Sea and the role of fishing, with clear tensions between economic interests, the livelihoods of fishers, and environmental protection.

Here are the key points and differing perspectives:

1.  **Importance of Fishing and Fishermen's Livelihoods:**
    *   **Sebastian Everding (The Left):** Acknowledges compromises in the report, emphasizing the need to find solutions *with* coastal regions and fishers. He stresses that fishing is not only a cultural and historical treasure but also economically vital for the Baltic Sea region. He believes fishers must be part of the solution, are already cooperating with science (e.g., stocking measures), and deserve EU support. He even notes the inclusion of "predators" in the discussion as a necessary, albeit difficult, compromise for conservationists.
    *   **Volker Schnurrbusch (ESN):** Expresses concern that the report's recommendations will further accelerate the decline of Baltic Sea fishing. He argues that more restrictions, uncertainty, and pressure on fishers will lead to less European fish and more imports from countries with lower standards, which he deems unsustainable. He highlights that fishers already suffer from low quotas, bureaucracy, and falling incomes, criticizing Brussels' tendency for more centralization and regulation.

2.  **Environmental Concerns and Critiques of Current Fishing Practices:**
    *   **An unnamed speaker (implied to be from The Left, but with a strong environmental focus):** Paints a grim picture of the Baltic Sea, citing "dead zones" from chemical and livestock industries, and marine life entangled in nets. This speaker vehemently criticizes industrial ships for catching fish not for human consumption, but for "fishmeal for fish farming and livestock industry." They advocate for immediately stopping the catching of endangered species and halting pollution. The speaker labels the idea of shooting seals and cormorants as a "bizarre plan" and an easy way to shift blame from human actions. They call for binding rules, clear reporting for lost gear, take-back systems, and an end to environmentally harmful materials, asserting that the Baltic Sea is "not a raw material warehouse."
    *   **Isabella Lövin (Rapporteur):** Emphasizes the need for a healthy, living Baltic Sea for swimming, food, and thriving coastal communities. She declares that "the time for business as usual is over" and demands that fishing should be for "food on the plate, not for feed for animal factories." Crucially, she calls for a "pause for industrial trawling until recovery is clear and lasting."

3.  **Policy and Reform:**
    *   **Emma Wiesner (Renew):** Asks Sebastian Everding if he supports the strong signal to *review the Baltic Sea map*, despite his group's initial reservations about the report due to the predator issue.
    *   **Sebastian Everding's response:** Agrees that "something absolutely has to change" and that the current situation cannot continue. He dismisses the discussion around predators as a "sham discussion" and emphasizes that the "complete fishing industry must be reformed" and the "complete handling of the Baltic Sea must change."
    *   **Commissioner Sinkevičius:** Acknowledges the importance of fishing and the challenges. He mentions that the Commission has initiated a reflection process to enhance the functioning of multiannual management plans and is in close contact with ICES (International Council for the Exploration of the Sea) and Member States to improve scientific advice. He refers to a decision in October Council for working towards a "rebuilding trajectory for our fisheries."

In essence, the discussion highlights a deep division: one side prioritizes the continuation and support of the traditional fishing industry, seeing fishers as part of the solution and fearing over-regulation. The other side calls for radical reform, blaming industrial fishing practices, pollution, and the use of fish for animal feed as major contributors to the Baltic Sea's degradation, and views the focus on predators as a distraction from human responsibility. All parties, however, seem to agree that the Baltic Sea is in a critical state and action is needed.

That's not bad, but we could do better by writing a more extensive prompt, giving the model more guidance.

It turns out we're not the first ones doing this, and LangChain has a library of pre-made prompts for us.

👉 Run the cell below, and try to understand how it works. (You'll get a warning about LangSmithMissingAPIKeyWarning, you can disregard that.)

In [48]:
from langchain_classic import hub

prompt_template = hub.pull("rlm/rag-prompt")

example_messages = prompt_template.invoke(
    {"context": "(context goes here)", "question": "(question goes here)"}
).to_messages()

print("\n")
print(example_messages[0].content)



You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: (question goes here) 
Context: (context goes here) 
Answer:


See how LangChain generated a more precise prompt for us? Let's use this for our RAG!

👉 First, join all retrieved docs into one long string, separated by two newlines.

In [50]:
# YOUR CODE HERE
docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

👉 Next, create a `prompt` starting from your query and the retrieved documents. Remember to look at the example above.

In [51]:
# YOUR CODE HERE
the_prompt = prompt_template.invoke(
    {"context": docs_content, "question": query}
)

👉 Finally use the LLM model with `the_prompt` we just created:

In [52]:
# YOUR CODE HERE
response = llm.invoke(the_prompt)

Markdown(response.content)

The discussion highlights the significant cultural and economic importance of fishing in the Baltic Sea, noting concerns about its decline due to overfishing, pollution, and insufficient management plans. There is a strong call for reform, emphasizing that fishers must be included in developing solutions and advocating for a review of the Baltic Sea's multiannual management plan. Proposed measures include halting industrial trawling, addressing pollution, and ensuring fishing primarily serves human consumption rather than animal feed production.

🎉 We have finished our first RAG: the LLM generated text ***grounded*** in the documents we provided it with.

## 💾 Persisting our embeddings

So far we worked with an in-memory vector store. So when you will close your notebook, you will also loose all the embeddings.

⚠️ Remember that these embeddings are generated by models running on your provider's platform, in this case on Google's machines. And they don't work for free. 💰

For one, relatively small document like this one, the cost is low, but it quickly adds up. So far, we just workend on one day's transcripts. There are 3 more per session, 12 sessions per year, multiple years...

To solve this we will just replace our vector store by a persistent one. That's the advantage of LangChain: it's very easy to replace components.

Our in-memory vector store was great for experimenting, now we'll switch it for another one. We will use [Chroma](https://www.trychroma.com/), a very popular vector store. We can run it locally, and use it through LangChain.

We'll recreate our whole flow. It's a good exercise to try to bring it all together again in a couple of code cells. At the same time we'll refactor everything into some reusable code.

We want to have two functions in the end:

1. `embed_and_store()`: Add another session's transcript to our vector db, so that we have more data to retrieve from.
2. `answer()`: Query our vector store with different questions.

#### 1. Instantiate a Chroma vector store

👉 Look at [LangChain's documentation](https://python.langchain.com/docs/integrations/vectorstores/chroma/) to see how to create Chroma vector store **with data persistence** (i.e. storing the data in a directory on disk).

In [53]:
# YOUR CODE HERE
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="european_parliament",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

In [ ]:
all_splits

#### 2. Create `embed_and_store()`

👉 Complete the code for this function:

In [ ]:
def embed_and_store(file_path, vector_store):
    """Load a PDF file, split it into chunks, and store the chunks in a vector store."""
    # Load the PDF file
    # YOUR CODE HERE
    loader = PyPDFLoader(file_path, mode="single")
    pdf = loader.load()

    # Split the pages into chunks
    # YOUR CODE HERE
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=400
    )
    all_splits = text_splitter.split_documents(doc)

    # # Add the session_date to the metadata

    # Add the chunks to the vector store
    # YOUR CODE HERE
    document_ids = vector_store.add_documents(documents=all_splits)
    return document_ids

👉 Try out your function with a file or even two:

In [70]:
# YOUR CODE HERE

document_ids = embed_and_store(file_path, vector_store)

document_ids[:3]

['c85a6913-dd85-4b26-8bff-fc2949c1a39d',
 'f0073a02-d00b-4211-ae46-58495d494e0b',
 '28be4237-bc32-4964-b6db-272b5021c9c1']

#### 3. Create `answer()`

👉 Complete the code for this function:

In [67]:
def answer(query, vector_store, llm, prompt_template=None):
    """Answer a query using the vector store and the language model."""
    # Retrieve similar documents from the vector store
    retrieved_docs = vector_store.similarity_search(query, k=6)

    # Create the prompt
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # If no prompt template is provided, use the default one
    if not prompt_template:
        prompt_template = hub.pull("rlm/rag-prompt")

    prompt = prompt_template.invoke(
        {"context": docs_content, "question": query}
    )

    # Get the answer from the language model
    answer = llm.invoke(prompt)

    return answer.content

👉 Try out your function with a query of your liking:

In [68]:
# YOUR CODE HERE
query = 'Summerize the discussion about fishing'

rag_answer = answer(query, vector_store, llm)

Markdown(rag_answer)

The discussion highlights the critical state of the Baltic Sea due to overfishing and pollution, with calls for urgent action to reduce fishing pressure and reform fishery management. Participants debate whether current EU regulations are harming small-scale European fishermen and if predator control distracts from human overfishing. Ultimately, there is a shared goal for a healthy Baltic ecosystem that supports coastal communities, emphasizing collaboration with fishermen and scientific experts.

🏁 Congratulations! You now master RAG using LangChain, and you learned how to make reusable functions to add more documents to your vector store, and to query it.

## [Optional] Adding metadata

The RAG we set up queries all the documents from the vector store. Imagine we have multiple year's information in there. It would be handy if we could filter on years, or dates, no?

How to do that? Remember that the documents in the vector store contain metadata. If we could add the date to it, we could use it later to filter.

Tip: Add your metadata as early as possible in your pipeline. Don't try to add it after your data was already stored to the vector store.

👉 Adapt your `embed_and_store()` function.

In [ ]:
def embed_and_store_fancy(file_path, vector_store, session_date):
    """Load a PDF file, split it into chunks, and store the chunks in a vector store.
    Session_date is added to the metadata of each chunk."""
    pass  # YOUR CODE HERE

    return document_ids

👉 Try out your function and check that your vector store contains the extra metadata.

In [ ]:
# YOUR CODE HERE

Now we have to limit the retriever to the date asked by the user. 

👉 Adapt your `answer()` function so it can take a date and filter documents based on the new metadata.

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

Nice! You have combined similarity search with metadata search to create a powerful RAG system!